# UK Road Accident Severity Prediction
## 01 — Collision Data Preparation

This notebook prepares the UK Department for Transport collision dataset.

Objectives:
- Understand the dataset structure.
- Assess data quality.
- Clean the collision data.
- Control memory usage during processing.
- Export the cleaned dataset for integration.

Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

Load Dataset

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/UK-Accident-Prediction/dft-road-casualty-statistics-collision-1979-latest-published-year.csv', nrows=5)

In [ ]:
df.head()

,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
0,2018450290294,2018,450290294,487906,132932,-0.746206,51.088970,45,3,2,...,0,0,1,2,2,E01030927,3,1,0.000000,1.000000
1,2020230984311,2020,230984311,433493,258232,-1.511123,52.221299,23,2,5,...,0,13,2,1,1,E01031199,6,1,1.000000,0.000000
2,1998929800391,1998,929800391,391100,803100,NaN,NaN,92,3,1,...,0,0,2,-1,-1,-1,-1,-1,NaN,NaN
3,1996950661096,1996,950661096,331410,673460,NaN,NaN,95,3,2,...,0,0,1,-1,-1,-1,-1,-1,NaN,NaN
4,2006620156303,2006,620156303,300660,202130,-3.439174,51.709037,62,3,2,...,0,0,1,1,-1,W01001150,-1,0,0.021371,0.978629


In [ ]:
df.shape

(5, 44)

Filter Recent Records

The original dataset contains collision records from 1979 onwards.

To develop a model that better represents current road conditions, legislation and vehicle technologies, only collisions from **2010 onwards** are retained.

This still provides a large number of observations while improving the relevance of the training data.

In [ ]:
chunks = pd.read_csv('/content/drive/MyDrive/UK-Accident-Prediction/dft-road-casualty-statistics-collision-1979-latest-published-year.csv',
                     chunksize=100_000, dtype={'collision_index': 'string', 'collision_ref_no': 'string'})

In [ ]:
df = pd.concat([chunk.loc[chunk['collision_year'] >= 2010] for chunk in chunks], ignore_index=True)

In [ ]:
df.shape

(1886746, 44)

In [ ]:
df.memory_usage(deep=True).sum() / 1024**2

np.float64(1354.1901960372925)

Exploration and Data Cleaning

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1886746 entries, 0 to 1886745
Data columns (total 44 columns):
 #   Column                                            Dtype  
---  ------                                            -----  
 0   collision_index                                   string 
 1   collision_year                                    int64  
 2   collision_ref_no                                  string 
 3   location_easting_osgr                             float64
 4   location_northing_osgr                            float64
 5   longitude                                         float64
 6   latitude                                          float64
 7   police_force                                      int64  
 8   collision_severity                                int64  
 9   number_of_vehicles                                int64  
 10  number_of_casualties                              int64  
 11  date                                              object 
 12  

In [ ]:
df.columns

Index(['collision_index', 'collision_year', 'collision_ref_no',
       'location_easting_osgr', 'location_northing_osgr', 'longitude',
       'latitude', 'police_force', 'collision_severity', 'number_of_vehicles',
       'number_of_casualties', 'date', 'day_of_week', 'time',
       'local_authority_district', 'local_authority_ons_district',
       'local_authority_highway', 'local_authority_highway_current',
       'first_road_class', 'first_road_number', 'road_type', 'speed_limit',
       'junction_detail_historic', 'junction_detail', 'junction_control',
       'second_road_class', 'second_road_number',
       'pedestrian_crossing_human_control_historic',
       'pedestrian_crossing_physical_facilities_historic',
       'pedestrian_crossing', 'light_conditions', 'weather_conditions',
       'road_surface_conditions', 'special_conditions_at_site',
       'carriageway_hazards_historic', 'carriageway_hazards',
       'urban_or_rural_area', 'did_police_officer_attend_scene_of_accident',
 

In [ ]:
df.describe()

,collision_year,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,number_of_casualties,day_of_week,...,special_conditions_at_site,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
count,1.886746e+06,1.886545e+06,1.886545e+06,1.886545e+06,1.886545e+06,1.886746e+06,1.886746e+06,1.886746e+06,1.886746e+06,1.886746e+06,...,1.886746e+06,1.886746e+06,1.886746e+06,1.886746e+06,1.886746e+06,1.886746e+06,1.886746e+06,1.886746e+06,1.886746e+06,1.886746e+06
mean,2.016348e+03,4.489366e+05,2.862676e+05,-1.300337e+00,5.246391e+01,2.916419e+01,2.803487e+00,1.832587e+00,1.314067e+00,4.116988e+00,...,1.423610e-01,1.029614e-01,1.308200e+00,1.337305e+00,1.295016e+00,1.653465e+00,1.606470e-01,2.439496e-01,2.132975e-01,7.738560e-01
std,4.274201e+00,9.472039e+04,1.550009e+05,1.389491e+00,1.396045e+00,2.503738e+01,4.284728e-01,7.046588e-01,7.781080e-01,1.924024e+00,...,1.101660e+00,1.004069e+00,8.789282e+00,4.729997e-01,5.574166e-01,8.835930e-01,2.178901e+00,4.294628e-01,3.625177e-01,3.723175e-01
min,2.010000e+03,6.408400e+04,1.021100e+04,-7.525273e+00,4.991221e+01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,...,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.013000e+03,3.858040e+05,1.761700e+05,-2.213248e+00,5.147213e+01,6.000000e+00,3.000000e+00,1.000000e+00,1.000000e+00,2.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,2.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00,8.410091e-01
50%,2.016000e+03,4.541500e+05,2.337750e+05,-1.200037e+00,5.198953e+01,3.000000e+01,3.000000e+00,2.000000e+00,1.000000e+00,4.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,2.000000e+00,-1.000000e+00,0.000000e+00,4.187391e-02,9.544244e-01
75%,2.020000e+03,5.277810e+05,3.897360e+05,-1.584600e-01,5.340163e+01,4.500000e+01,3.000000e+00,2.000000e+00,1.000000e+00,6.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,1.000000e+00,2.000000e+00,-1.000000e+00,0.000000e+00,1.479813e-01,1.000000e+00
max,2.024000e+03,6.553910e+05,1.209512e+06,1.759829e+00,6.076372e+01,9.900000e+01,3.000000e+00,6.700000e+01,9.300000e+01,7.000000e+00,...,9.000000e+00,9.000000e+00,9.900000e+01,3.000000e+00,3.000000e+00,2.000000e+00,7.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


In [ ]:
df.isnull().sum()

,0
collision_index,0
collision_year,0
collision_ref_no,0
location_easting_osgr,201
location_northing_osgr,201
longitude,201
latitude,201
police_force,0
collision_severity,0
number_of_vehicles,0


In [ ]:
((df.isnull().sum() / len(df)) * 100).sort_values(ascending=False).map("{:.2f}%".format)

,0
local_authority_highway_current,0.02%
location_easting_osgr,0.01%
longitude,0.01%
latitude,0.01%
location_northing_osgr,0.01%
speed_limit,0.00%
collision_ref_no,0.00%
collision_year,0.00%
collision_severity,0.00%
number_of_vehicles,0.00%


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.nunique().sort_values()

,0
collision_injury_based,2
collision_severity,3
trunk_road_flag,3
did_police_officer_attend_scene_of_accident,4
urban_or_rural_area,4
pedestrian_crossing_human_control_historic,5
first_road_class,6
light_conditions,6
enhanced_severity_collision,6
road_type,7


In [ ]:
df.drop(
    columns=[
        'collision_ref_no',
        'enhanced_severity_collision',
        'collision_injury_based',
        'collision_adjusted_severity_serious',
        'collision_adjusted_severity_slight'
    ], inplace=True
)

In [ ]:
df.shape

(1886746, 39)

In [ ]:
columns = [
    'junction_detail_historic',
    'junction_detail',
    'pedestrian_crossing_human_control_historic',
    'pedestrian_crossing_physical_facilities_historic',
    'pedestrian_crossing',
    'carriageway_hazards_historic',
    'carriageway_hazards'
]

In [ ]:
for columns in columns:
  print(f'\n{columns}')
  print(df[columns].value_counts(dropna=False).sort_index())


junction_detail_historic
junction_detail_historic
-1      59621
 0     735304
 1     154165
 2      23962
 3     557863
 5      25217
 6     176252
 7      21873
 8      56415
 9      63464
 99     12610
Name: count, dtype: int64

junction_detail
junction_detail
-1      17684
 0     967476
 13    572184
 16    181409
 17     22675
 18     57359
 19     67957
 99         2
Name: count, dtype: int64

pedestrian_crossing_human_control_historic
pedestrian_crossing_human_control_historic
-1      63920
 0    1774567
 1       5675
 2      12655
 9      29929
Name: count, dtype: int64

pedestrian_crossing_physical_facilities_historic
pedestrian_crossing_physical_facilities_historic
-1      63830
 0    1440525
 1      63806
 4     104419
 5     139796
 7       4974
 8      42658
 9      26738
Name: count, dtype: int64

pedestrian_crossing
pedestrian_crossing
-1        5477
 0     1489758
 11       5852
 12      13109
 13      62597
 14     103106
 15     137649
 16       5002
 17      43426
 9

In [ ]:
df.drop(
    columns=[
        'junction_detail_historic',
        'pedestrian_crossing_human_control_historic',
        'pedestrian_crossing_physical_facilities_historic',
        'carriageway_hazards_historic'
    ],
    inplace=True
)

df.shape

(1886746, 35)

In [ ]:
df[
    [
        'location_easting_osgr',
        'location_northing_osgr',
        'longitude',
        'latitude',
        'local_authority_highway_current',
        'speed_limit'
    ]
].isna().sum()

,0
location_easting_osgr,201
location_northing_osgr,201
longitude,201
latitude,201
local_authority_highway_current,287
speed_limit,37


In [ ]:
df[['date', 'time']].head(10)

,date,time
0,15/04/2018,08:00
1,21/09/2020,23:00
2,29/04/2019,09:01
3,05/06/2011,13:05
4,23/01/2013,11:00
5,18/10/2011,21:50
6,10/01/2014,12:20
7,21/02/2022,16:40
8,30/07/2019,08:01
9,05/04/2019,20:00


In [ ]:
df['speed_limit'].value_counts(dropna=False).sort_index()

,count
speed_limit,
-1.0,95
0.0,1
10.0,5
15.0,1
20.0,134126
30.0,1145813
40.0,158290
50.0,75049
60.0,252453


In [ ]:
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y', errors='coerce')
df['time'] = pd.to_datetime(df['time'], format='%H:%M', errors='coerce')

In [ ]:
print(df['date'].isna().sum())
print(df['time'].isna().sum())

0
0


In [ ]:
df['collision_severity'].value_counts().sort_index()

,count
collision_severity,
1,24238
2,322295
3,1540213


In [ ]:
df['collision_severity'].value_counts(normalize=True).sort_index() * 100

,proportion
collision_severity,
1,1.284646
2,17.082056
3,81.633299


Save Clean Dataset

Export the cleaned dataset.

The resulting dataset will be used during the data integration stage together with the vehicle and casualty datasets.

In [ ]:
df.to_csv('/content/drive/MyDrive/UK-Accident-Prediction/collision_clean.csv', index=False)

The collision dataset was successfully prepared for the next stage of the project.

Summary of the work completed:

- Filtered the dataset to include collisions from 2010 onwards.
- Assessed the overall quality of the data.
- Verified that no duplicate records were present.
- Analysed missing values.
- Removed redundant identifier variables.
- Removed historical variables replaced by newer standards.
- Removed variables that could introduce data leakage into the machine learning model.
- Imputed missing values where appropriate.
- Exported a clean dataset ready for data integration.

In [35]:
print('Collision Dataset Summary')
print('='*50)
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')
print(f'Missing Values: {df.isnull().sum().sum()}')
print(f'Duplicate Rows: {df.duplicated().sum()}')

Collision Dataset Summary
Rows: 1886746
Columns: 35
Missing Values: 1128
Duplicate Rows: 0
